In this notebook we show how to simulate the seizure spread and controllability.

In [ ]:
import numpy as np
import math
from numba import jit
import time
from datetime import datetime
import os
import sys
from normalization import normalize
import scipy.io
from functions import EZs, x0_w_init
from Dynamics_ctr6D_z import Dynamics_ctr6D_z
from file_loading import *
from functions import Jacobian6D , Xprime
from scipy.optimize import fsolve

In [ ]:
# Simulation parameters-------------------------------
patient_name='P1'
EZ_ind=0
noise_strength=0.02
ctrw=0.75 # control gain
ctrD=10000 # hard bound control duration 
ctrlag=1 # control lag in seconds afeter seizure onset in the EZ node
x0=-2.15 # Network excitability
w=0.5 # global coupling weight

# Loading the connectivity Network -------------------------
connectivity_file="connectivity_"+patient_name+".zip"
network_directory="./data/"
W=get_data(network_directory+connectivity_file,'weights.txt')
W_normal=normalize(W,"global") # Normalized connectivity weight
## W_altered=normalize(W,"global")
N=W_normal.shape[0]
h=0.05
speed=60.0
Tau_normal=get_data(network_directory+connectivity_file,'tract_lengths.txt')/speed
Tau=(Tau_normal/h).astype(int)
t0=np.max(Tau)
simulation_time=30000.0
t_EZ_onset=6000.0
EZ_x0=-1.8
M=t0+int(simulation_time/h)
M0=t0+int(t_EZ_onset/h)
T_start=time.time()
EZ_areas=EZs(patient_name)
EZ=np.array([EZ_areas[EZ_ind]])
## W_altered[EZ,57:N]=0
## W_altered[57:N,EZ]=0


# Setting random seed---------------------------------
seed=int(1000*time.time())
seed =np.mod(seed,2**31)+1

In [ ]:
# Simulating Uncontrolled network





# Building libraries to pass as input to the simulator-------------------------

Network={
    "h":h,
    "Tau":Tau,
    "W":W_normal * w,
    "N":N,
    "t0":t0,
    "M":M,
    "EZs":EZ
}

epileptor_6D={
    "model_name":"epileptor",
    "model_dimension":6,
    "I1" : 3.1,  
    "I2" : 0.45,  
    "r" : 0.00015,  
    "tau2" : 10.0,   
    "gamma" : 0.01,
    "m":0.00,
    "x0":np.ones(N)*x0
}


wW=W_normal * w

X0=np.zeros(6*N)
X0[ 0*N:1*N ]=-1.62
X0[ 1*N:2*N ]=-12.2
X0[ 2*N:3*N ]=3.1
X0[ 3*N:4*N ]=-0.844
X0[ 4*N:5*N ]=0
X0[ 5*N:6*N ]=-162
# Solve for the fixed point
Fixed_point=scipy.optimize.fsolve(Xprime,X0,args=(wW,epileptor_6D))
# 
print('fixed point close to root',np.max(Xprime(Fixed_point,wW,epileptor_6D)))


# calculate the preictal Jacobian and its eigenvalues ------------------------------
J=Jacobian6D(Fixed_point,wW,epileptor_6D)
eigvals,_=np.linalg.eig(J)
print('real part of largest eigenvalue of Jacobian: ',np.max(eigvals.real))

# Selecting nodes for control actuation
Ktr_nodes=[EZ]
print('Nodes to be controlled: ',Ktr_nodes)

# Setting up control gain values

Ktr_W=1*np.zeros(6)
Ktr_W[0]=ctrw
Ktr_W[3]=ctrw

X_str=np.zeros([6,N])
X_str[0,:]=Fixed_point[ 0*N:1*N ]
X_str[1,:]=Fixed_point[ 1*N:2*N ]
X_str[2,:]=Fixed_point[ 2*N:3*N ]
X_str[3,:]=Fixed_point[ 3*N:4*N ]
X_str[4,:]=Fixed_point[ 4*N:5*N ]
X_str[5,:]=Fixed_point[ 5*N:6*N ]
Ktr_D=ctrD*50
Ktr_lag=ctrlag*50
Ktr=np.zeros(N)
Ktr[Ktr_nodes]=1

CTRL={
    "Ktr":Ktr,
    "Ktr_D":Ktr_D,
    "Ktr_lag":Ktr_lag,
    "X_str":X_str,
    "Ktr_W":Ktr_W
}


# Running the simulation---------------------------------------------
# creating a simulation object from the Dynamics class
simu=Dynamics_ctr6D_z(epileptor_6D,"Runge_Kutta_2",Network,CTRL)
# setting the initial conditions for the 6 variabls of the dynamics 
# sim.Set_Initial_Condition([ x1 , y1 , z , x2 , y2 , g ])
simu.Set_Initial_Condition([ -1.6 , -12.0 , 3.2 , 0.00 , 0.00 , -160 ])
# set the noise (Same convention as initial conditions)
simu.Set_noise([ 0.0 , 0.0 , 0.0 , noise_strength  , noise_strength  , 0.0 ] , seed)
# forcing refractory state after seizure termination
simu.refractory=True
# control on/off
simu.Ktr_flag=False
# running the simulation
simu.run(t_EZ_onset,EZ_x0) 

In [ ]:
# Simulating Controlled network Only EZ node

# Building libraries to pass as input to the simulator-------------------------

Network={
    "h":h,
    "Tau":Tau,
    "W":W_normal * w,
    "N":N,
    "t0":t0,
    "M":M,
    "EZs":EZ
}

epileptor_6D={
    "model_name":"epileptor",
    "model_dimension":6,
    "I1" : 3.1,  
    "I2" : 0.45,  
    "r" : 0.00015,  
    "tau2" : 10.0,   
    "gamma" : 0.01,
    "m":0.00,
    "x0":np.ones(N)*x0
}


wW=W_normal * w

X0=np.zeros(6*N)
X0[ 0*N:1*N ]=-1.62
X0[ 1*N:2*N ]=-12.2
X0[ 2*N:3*N ]=3.1
X0[ 3*N:4*N ]=-0.844
X0[ 4*N:5*N ]=0
X0[ 5*N:6*N ]=-162
# Solve for the fixed point
Fixed_point=scipy.optimize.fsolve(Xprime,X0,args=(wW,epileptor_6D))
# 
print('fixed point close to root',np.max(Xprime(Fixed_point,wW,epileptor_6D)))


# calculate the preictal Jacobian and its eigenvalues ------------------------------
J=Jacobian6D(Fixed_point,wW,epileptor_6D)
eigvals,_=np.linalg.eig(J)
print('real part of largest eigenvalue of Jacobian: ',np.max(eigvals.real))

# Selecting nodes for control actuation
Ktr_nodes=[EZ]
print('Nodes to be controlled: ',Ktr_nodes)

# Setting up control gain values

Ktr_W=1*np.zeros(6)
Ktr_W[0]=ctrw
Ktr_W[3]=ctrw

X_str=np.zeros([6,N])
X_str[0,:]=Fixed_point[ 0*N:1*N ]
X_str[1,:]=Fixed_point[ 1*N:2*N ]
X_str[2,:]=Fixed_point[ 2*N:3*N ]
X_str[3,:]=Fixed_point[ 3*N:4*N ]
X_str[4,:]=Fixed_point[ 4*N:5*N ]
X_str[5,:]=Fixed_point[ 5*N:6*N ]
Ktr_D=ctrD*50
Ktr_lag=ctrlag*50
Ktr=np.zeros(N)
Ktr[Ktr_nodes]=1

CTRL={
    "Ktr":Ktr,
    "Ktr_D":Ktr_D,
    "Ktr_lag":Ktr_lag,
    "X_str":X_str,
    "Ktr_W":Ktr_W
}


# Running the simulation---------------------------------------------
# creating a simulation object from the Dynamics class
simc=Dynamics_ctr6D_z(epileptor_6D,"Runge_Kutta_2",Network,CTRL)
# setting the initial conditions for the 6 variabls of the dynamics 
# sim.Set_Initial_Condition([ x1 , y1 , z , x2 , y2 , g ])
simc.Set_Initial_Condition([ -1.6 , -12.0 , 3.2 , 0.00 , 0.00 , -160 ])
# set the noise (Same convention as initial conditions)
simc.Set_noise([ 0.0 , 0.0 , 0.0 , noise_strength  , noise_strength  , 0.0 ] , seed)
# forcing refractory state after seizure termination
simc.refractory=True
# control on/off
simc.Ktr_flag=True
# running the simulation
simc.run(t_EZ_onset,EZ_x0) 

In [ ]:
# Simulating Controlled network EZ + 5 non-EZ nodes



# Building libraries to pass as input to the simulator-------------------------

Network={
    "h":h,
    "Tau":Tau,
    "W":W_normal * w,
    "N":N,
    "t0":t0,
    "M":M,
    "EZs":EZ
}

epileptor_6D={
    "model_name":"epileptor",
    "model_dimension":6,
    "I1" : 3.1,  
    "I2" : 0.45,  
    "r" : 0.00015,  
    "tau2" : 10.0,   
    "gamma" : 0.01,
    "m":0.00,
    "x0":np.ones(N)*x0
}


wW=W_normal * w

X0=np.zeros(6*N)
X0[ 0*N:1*N ]=-1.62
X0[ 1*N:2*N ]=-12.2
X0[ 2*N:3*N ]=3.1
X0[ 3*N:4*N ]=-0.844
X0[ 4*N:5*N ]=0
X0[ 5*N:6*N ]=-162
# Solve for the fixed point
Fixed_point=scipy.optimize.fsolve(Xprime,X0,args=(wW,epileptor_6D))
# 
print('fixed point close to root',np.max(Xprime(Fixed_point,wW,epileptor_6D)))


# calculate the preictal Jacobian and its eigenvalues ------------------------------
J=Jacobian6D(Fixed_point,wW,epileptor_6D)
eigvals,_=np.linalg.eig(J)
print('real part of largest eigenvalue of Jacobian: ',np.max(eigvals.real))

# Selecting nodes for control actuation

# Solving for fixed point in the presence of active EZ--------------------------------
excitability=np.ones(N)*x0
excitability[EZ]=-1.8

epileptor_6DEZ={
    "model_name":"epileptor",
    "model_dimension":6,
    "I1" : 3.1,  
    "I2" : 0.45,  
    "r" : 0.00015,  
    "tau2" : 10.0,   
    "gamma" : 0.01,
    "m":0.00,
    "x0":excitability
}

# Setting the initial condition for fsolve function
X0=np.zeros(6*N)
X0[ 0*N:1*N ]=-1.62
X0[ 1*N:2*N ]=-12.2
X0[ 2*N:3*N ]=3.1
X0[ 3*N:4*N ]=-0.844
X0[ 4*N:5*N ]=0
X0[ 5*N:6*N ]=-162
# Solve for the fixed point
Fixed_pointEZ=scipy.optimize.fsolve(Xprime,X0,args=(wW,epileptor_6DEZ))
# 
print('fixed point EZ close to root',np.max(Xprime(Fixed_pointEZ,wW,epileptor_6DEZ)))
        
# calculate the Jacobian and its eigenvalues in the presence of active EZ---------------
JEZ=Jacobian6D(Fixed_pointEZ,wW,epileptor_6DEZ)
eigvalsEZ,eigvectorsEZ=np.linalg.eig(JEZ)
eigrealEZ=eigvalsEZ.real
print('real part of largest eigenvalue of Jacobian with EZ: ',np.max(eigvalsEZ.real))


#### Nodes to be controlled based on leading eigenvector 
ind1=np.argsort(eigrealEZ)[::-1]
i_lead=ind1[0]
V_z=np.abs(eigvectorsEZ[2*N:3*N,i_lead])
ind2=np.argsort(V_z)[::-1]
Ktr_nodes=ind2[0:5+1] # In ind2[i:j+1], j is the number of non-EZ nodes. Set i=0 to include the EZ i=1 to exclude the EZ.  
print('Nodes to be controlled: ',Ktr_nodes)

# Setting up control gain values

Ktr_W=1*np.zeros(6)
Ktr_W[0]=ctrw
Ktr_W[3]=ctrw

X_str=np.zeros([6,N])
X_str[0,:]=Fixed_point[ 0*N:1*N ]
X_str[1,:]=Fixed_point[ 1*N:2*N ]
X_str[2,:]=Fixed_point[ 2*N:3*N ]
X_str[3,:]=Fixed_point[ 3*N:4*N ]
X_str[4,:]=Fixed_point[ 4*N:5*N ]
X_str[5,:]=Fixed_point[ 5*N:6*N ]
Ktr_D=ctrD*50
Ktr_lag=ctrlag*50
Ktr=np.zeros(N)
Ktr[Ktr_nodes]=1

CTRL={
    "Ktr":Ktr,
    "Ktr_D":Ktr_D,
    "Ktr_lag":Ktr_lag,
    "X_str":X_str,
    "Ktr_W":Ktr_W
}


# Running the simulation---------------------------------------------
# creating a simulation object from the Dynamics class
simcn=Dynamics_ctr6D_z(epileptor_6D,"Runge_Kutta_2",Network,CTRL)
# setting the initial conditions for the 6 variabls of the dynamics 
# sim.Set_Initial_Condition([ x1 , y1 , z , x2 , y2 , g ])
simcn.Set_Initial_Condition([ -1.6 , -12.0 , 3.2 , 0.00 , 0.00 , -160 ])
# set the noise (Same convention as initial conditions)
simcn.Set_noise([ 0.0 , 0.0 , 0.0 , noise_strength  , noise_strength  , 0.0 ] , seed)
# forcing refractory state after seizure termination
simcn.refractory=True
# control on/off
simcn.Ktr_flag=True
# running the simulation
simcn.run(t_EZ_onset,EZ_x0) 

In [ ]:
simcn.seizure_onsets[EZ]

In [ ]:
# Plotting uncontrolled and controlled time series 


import matplotlib.pyplot as plt
plt.figure(figsize=(30,20))
plt.subplot(1,3,1)
t1=3000
t2=5200
onset=simc.seizure_onsets[EZ]/50
time=simcn.T_axis[t1:t2]/50-onset
for i in range(N):
    if(i==EZ):
        plt.plot(time,0.2*simu.Y[t1:t2,i,0]+i+1,'-r')
    else:
        plt.plot(time,0.2*simu.Y[t1:t2,i,0]+i+1,'-k')

plt.xlabel('Time [s]')  

plt.ylabel('Node index')
plt.rcParams['font.size'] = '32'
plt.title('Uncontrolled')
print('seizure size =',np.sum(simu.seizure_01/2))


onset=simc.seizure_onsets[EZ]/50
time=simcn.T_axis[t1:t2]/50-onset
plt.subplot(1,3,2)
for i in range(N):
    if(i==EZ):
        plt.plot(time,0.2*simc.Y[t1:t2,i,0]+i+1,'-r')
    else:
        plt.plot(time,0.2*simc.Y[t1:t2,i,0]+i+1,'-k')

plt.xlabel('Time [s]')  
plt.title('Feedback on only EZ')
# plt.ylabel('Node index')
plt.rcParams['font.size'] = '32'
print('seizure size =',np.sum(simc.seizure_01/2))


onset=simcn.seizure_onsets[EZ]/50
time=simcn.T_axis[t1:t2]/50-onset
plt.subplot(1,3,3)
for i in range(N):
    if(Ktr[i]==1):
        plt.plot(time,0.2*simcn.Y[t1:t2,i,0]+i+1,'-r')
    else:
        plt.plot(time,0.2*simcn.Y[t1:t2,i,0]+i+1,'-k')

plt.xlabel('Time [s]') 
plt.title('Feedback on EZ + 5 non-EZ ')
# plt.ylabel('Node index')
plt.rcParams['font.size'] = '32'
print('seizure size =',np.sum(simcn.seizure_01/2))